# FHERMA — búsqueda automática acotada de configuraciones CUDA
Compara hasta 12 configuraciones de NTT y CRT. Descarta las que no compilan o fallan exactitud; mide en la GPU gratuita asignada por Colab. **No envía variantes al concurso ni garantiza 700 µs**.
En Colab, activa T4 GPU y selecciona Ejecutar todas. El experimento tiene un límite de 40 minutos y 12 compilaciones.


In [ ]:
import subprocess, re
r=subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],text=True,capture_output=True)
print(r.stdout or r.stderr)
if r.returncode: raise RuntimeError('GPU no asignada en Colab')
print(subprocess.run(['nvcc','--version'],text=True,capture_output=True).stdout[-300:])


In [ ]:
import pathlib, subprocess
root=pathlib.Path('/content/fherma-test')
if not (root/'.git').exists():
 subprocess.run(['git','clone','--depth','1','--branch','research-auto-tune-ntt-crt','https://github.com/aritomas3123-boop/fherma-gpu-polymul',str(root)],check=True)
else:
 subprocess.run(['git','-C',str(root),'fetch','origin','research-auto-tune-ntt-crt'],check=True)
 subprocess.run(['git','-C',str(root),'checkout','-B','research-auto-tune-ntt-crt','origin/research-auto-tune-ntt-crt'],check=True)
print('Experiment commit:',subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True))


In [ ]:
import urllib.request
oracle=root/'oracle.py'
if not oracle.exists(): urllib.request.urlretrieve('https://raw.githubusercontent.com/fherma-ai/polynomial-mult-cupqc-example/main/oracle.py',oracle)
subprocess.run(['python3',str(root/'autotune/grid.py'),'--max-builds','12','--max-minutes','40','--finalists','3'],check=True)
print('Resultados en /content/fherma-autotune-results.csv')
